# Phishing Website Detection: Exploratory Analysis & Modeling

This notebook walks through:
1. Loading the cached UCI Phishing Websites dataset (`11,055` websites, `30` features).
2. Checking class balance, missing values, and feature distributions.
3. Running the leak-free `60% / 20% / 20%` Train / Validation / Test split.
4. Tuning Logistic Regression, SVM, Decision Tree, and Naive Bayes on Validation and evaluating on Test.
5. Inspecting error analysis (False Positives vs. False Negatives).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, get_dataset_summary, TARGET_COLUMN
from src.preprocessing import split_dataset
from src.train_evaluate import run_training_pipeline

In [ ]:
df = load_dataset()
summary = get_dataset_summary(df)
print("Dataset Summary:", summary)
df.head()

In [ ]:
splits = split_dataset(df, random_state=42)
print(f"Train: {len(splits.X_train)}, Val: {len(splits.X_val)}, Test: {len(splits.X_test)}")
print("Train phishing rate:", round(splits.y_train.mean(), 4))
print("Val phishing rate:", round(splits.y_val.mean(), 4))
print("Test phishing rate:", round(splits.y_test.mean(), 4))

In [ ]:
# Top correlations with target (1 = Phishing, 0 = Legitimate)
encoded_df = splits.X_train.copy()
encoded_df["Phishing"] = splits.y_train
corrs = encoded_df.corr()["Phishing"].drop("Phishing").sort_values()
corrs.head(10)

In [ ]:
report = run_training_pipeline(random_state=42, save_artifacts=True)
print("Best model (selected on Validation):", report["best_model_name"])
pd.DataFrame(report["test_results"]).T[["precision", "recall", "f1", "roc_auc", "pr_auc", "accuracy"]]

In [ ]:
print("Error Analysis Summary:")
report["error_analysis"]